In [1]:
%pip install -q boto3 s3fs pandas pyarrow requests tqdm pyspark

Note: you may need to restart the kernel to use updated packages.


In [4]:
import os

os.environ["AWS_REGION"] = "us-east-1"
os.environ["AWS_DEFAULT_REGION"] = "us-east-1"
# opcional pero recomendable para evitar intentos a IMDS
os.environ["AWS_EC2_METADATA_DISABLED"] = "true"

In [5]:
import os
from pyspark.sql import SparkSession

# ---- AWS/MinIO ----
AWS_REGION = "us-east-1"
MINIO_ENDPOINT = "http://minio:9000"

AWS_ACCESS_KEY_ID = os.getenv("MINIO_ROOT_USER", "minioadmin")
AWS_SECRET_ACCESS_KEY = os.getenv("MINIO_ROOT_PASSWORD", "minioadmin123")

os.environ["AWS_REGION"] = AWS_REGION
os.environ["AWS_DEFAULT_REGION"] = AWS_REGION
os.environ["AWS_ACCESS_KEY_ID"] = AWS_ACCESS_KEY_ID
os.environ["AWS_SECRET_ACCESS_KEY"] = AWS_SECRET_ACCESS_KEY
os.environ["AWS_EC2_METADATA_DISABLED"] = "true"

# ---- Iceberg REST Catalog (Nessie) ----
CATALOG = "lk"
WAREHOUSE = "s3://lakehouse/warehouse"      
NESSIE_URI = "http://nessie:19120/iceberg"

spark = (
    SparkSession.builder
    .appName("nessie-iceberg-minio")

    # Dependencies
    .config(
        "spark.jars.packages",
        ",".join([
            "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.2",
            "org.apache.iceberg:iceberg-aws-bundle:1.5.2",
        ])
    )

    # --- Iceberg REST Catalog---
    .config(f"spark.sql.catalog.{CATALOG}", "org.apache.iceberg.spark.SparkCatalog")
    .config(f"spark.sql.catalog.{CATALOG}.type", "rest")
    .config(f"spark.sql.catalog.{CATALOG}.uri", NESSIE_URI)
    .config(f"spark.sql.catalog.{CATALOG}.warehouse", WAREHOUSE)

    # --- FileIO S3 (Iceberg) ---
    .config(f"spark.sql.catalog.{CATALOG}.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config(f"spark.sql.catalog.{CATALOG}.s3.endpoint", MINIO_ENDPOINT)
    .config(f"spark.sql.catalog.{CATALOG}.s3.path-style-access", "true")
    .config(f"spark.sql.catalog.{CATALOG}.s3.region", AWS_REGION)

    # ✅ force static credentials (Iceberg S3FileIO)
    .config(f"spark.sql.catalog.{CATALOG}.s3.access-key-id", AWS_ACCESS_KEY_ID)
    .config(f"spark.sql.catalog.{CATALOG}.s3.secret-access-key", AWS_SECRET_ACCESS_KEY)

    # ✅ Force S3A to use SimpleAWSCredentialsProvider
    .config("spark.hadoop.fs.s3a.endpoint", "minio:9000")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .config("spark.hadoop.fs.s3a.access.key", AWS_ACCESS_KEY_ID)
    .config("spark.hadoop.fs.s3a.secret.key", AWS_SECRET_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.endpoint.region", AWS_REGION)

    .getOrCreate()
)

spark.sql(f"SHOW NAMESPACES IN {CATALOG}").show(truncate=False)

+---------+
|namespace|
+---------+
|raw      |
+---------+



In [6]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS lk.raw2")
spark.sql("""
  CREATE TABLE IF NOT EXISTS lk.raw.smoke_py2 (
    id INT,
    val STRING
  )
  USING iceberg
""")
spark.sql("INSERT INTO lk.raw.smoke_py2 VALUES (1,'a'),(2,'b')")
spark.sql("SELECT * FROM lk.raw.smoke_py2").show()

+---+---+
| id|val|
+---+---+
|  1|  a|
|  2|  b|
+---+---+

